In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


In [2]:
spark = SparkSession.builder.appName("Silver Products Validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
bronze_products = spark.read.parquet("../data/bronze/products")
silver_products = spark.read.parquet("../data/silver/products")

In [4]:
bronze_products.count()

32951

In [5]:
silver_products.count()

32951

In [6]:
silver_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [7]:
from pyspark.sql.functions import count,col,when
silver_products.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in silver_products.columns
]).show(vertical=True)

-RECORD 0-------------------------
 product_id                 | 0   
 product_category_name      | 610 
 product_name_lenght        | 610 
 product_description_lenght | 610 
 product_photos_qty         | 610 
 product_weight_g           | 2   
 product_length_cm          | 2   
 product_height_cm          | 2   
 product_width_cm           | 2   



In [8]:
duplicate_silver_products= silver_products.groupBy("product_id").count().filter(col("count")>1)
duplicate_silver_products.show()
print("Duplicate silver products:",duplicate_silver_products.count())

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+

Duplicate silver products: 0


In [9]:
silver_products.filter(col("product_weight_g") < 0).count()

0

In [10]:
silver_products.filter(col("product_length_cm") < 0).count()

0

In [11]:
silver_products.filter(col("product_height_cm") < 0).count()

0

In [12]:
silver_products.filter(col("product_width_cm") < 0).count()

0

In [13]:
silver_products.filter(col("product_photos_qty") < 0).count()

0

In [14]:
silver_products.show(10,truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |1000            |30               |18               |20              |
|96bd76ec8810374ed1b65e291975717f|e